<a href="https://colab.research.google.com/github/karunaav/AI-Bookshelf-Scanner/blob/main/FINALPROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Final Project-Accessible Pedestrian Signals and Pedestrian Safety in NYC**

Karuna Venkatesh-Data Collection,Data Cleaning, Data preperation

Sabria Islam-

Seerat Hameed-Treatment data, spatial join, intersection-year panel

Ugne Morkute-

Wenjin Liu-


### Data Collection,Data Cleaning, Data preperation

Imports

## Person 1: Pedestrian Injury Data (Collection, Cleaning, Preparation)
This section prepares a clean file of pedestrian injuries in NYC from April 2016 to December 2023, which is used to build the intersection-year panel and to test H2 (pedestrians aged 65+ and nighttime injuries).

### 1. Setup
Import libraries and set the study period (April 1, 2016 to December 31, 2023).

In [1]:
import pandas as pd
import numpy as np

START, END = "2016-04-01", "2023-12-31"

### 2. Data collection
We download two NYC Open Data tables through the Socrata API: **Motor Vehicle Collisions – Crashes** (location and time of each crash) and **Motor Vehicle Collisions – Person** (one row per person involved). To keep files manageable, the download is filtered on the server to the study period, to crashes with at least one pedestrian injured or killed, and to pedestrians recorded as injured or killed.

In [6]:
!mkdir -p data
!wget -O data/crashes.csv "https://data.cityofnewyork.us/resource/h9gi-nx95.csv?\$select=collision_id,crash_date,crash_time,borough,latitude,longitude&\$where=crash_date%20between%20'2016-04-01T00:00:00'%20and%20'2023-12-31T23:59:59'%20AND%20(number_of_pedestrians_injured%3E0%20OR%20number_of_pedestrians_killed%3E0)&\$limit=500000"
!wget -O data/persons.csv "https://data.cityofnewyork.us/resource/f55k-p6yu.csv?\$select=collision_id,unique_id,person_id,person_type,person_injury,person_age,crash_date,crash_time&\$where=crash_date%20between%20'2016-04-01T00:00:00'%20and%20'2023-12-31T23:59:59'%20AND%20person_type='Pedestrian'%20AND%20person_injury%20in('Injured','Killed')&\$limit=500000"
!wc -l data/crashes.csv data/persons.csv

--2026-11-09 23:57:50--  https://data.cityofnewyork.us/resource/h9gi-nx95.csv?$select=collision_id,crash_date,crash_time,borough,latitude,longitude&$where=crash_date%20between%20'2016-04-01T00:00:00'%20and%20'2023-12-31T23:59:59'%20AND%20(number_of_pedestrians_injured%3E0%20OR%20number_of_pedestrians_killed%3E0)&$limit=500000
Resolving data.cityofnewyork.us (data.cityofnewyork.us)... 52.206.68.26, 52.206.140.205, 52.206.140.199
Connecting to data.cityofnewyork.us (data.cityofnewyork.us)|52.206.68.26|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/csv]
Saving to: ‘data/crashes.csv’

data/crashes.csv        [             <=>    ]   6.76M  2.39MB/s    in 2.8s    

2026-11-09 23:58:15 (2.39 MB/s) - ‘data/crashes.csv’ saved [7090634]

--2026-11-09 23:58:15--  https://data.cityofnewyork.us/resource/f55k-p6yu.csv?$select=collision_id,unique_id,person_id,person_type,person_injury,person_age,crash_date,crash_time&$where=crash_date%20between%20'2016-04

### 3. Load data
Load both tables, parse crash dates, and keep only records within the study period. We print the number of pedestrians per year and the injury types.

In [7]:
crashes = pd.read_csv("data/crashes.csv", low_memory=False)
persons = pd.read_csv("data/persons.csv", low_memory=False)

for d in (crashes, persons):
    d["crash_date"] = pd.to_datetime(d.crash_date.astype(str).str[:10], errors="coerce")

crashes = crashes[crashes.crash_date.between(START, END)]
persons = persons[persons.crash_date.between(START, END)]

print("Crashes:", len(crashes), "| Pedestrians:", len(persons))
print(persons.crash_date.dt.year.value_counts().sort_index())
print(persons.person_injury.value_counts(dropna=False))

Crashes: 71273 | Pedestrians: 74384
crash_date
2016     8444
2017    11278
2018    11246
2019    10699
2020     6793
2021     7634
2022     9108
2023     9182
Name: count, dtype: int64
person_injury
Injured    73407
Killed       977
Name: count, dtype: int64


### 4. Filter to injured or killed pedestrians
Standardize the injury labels, keep pedestrians recorded as injured or killed, and remove duplicate person records (same collision ID and person ID).

In [8]:
persons["person_injury"] = persons.person_injury.str.strip().str.title()
n0 = len(persons)
peds = persons[persons.person_injury.isin(["Injured", "Killed"])]
peds = peds.drop_duplicates(["collision_id", "person_id"])
print(f"Pedestrians: {n0:,} -> {len(peds):,} after injury filter and de-duplication")

Pedestrians: 74,384 -> 74,384 after injury filter and de-duplication


### 5. Join Person to Crashes
Join each pedestrian record to its crash on collision ID to add borough and coordinates, and combine the crash date and time into one timestamp.

In [9]:
crashes = crashes.drop_duplicates("collision_id")
df = peds.merge(crashes[["collision_id", "borough", "latitude", "longitude"]],
                on="collision_id", how="inner")

df["crash_dt"] = pd.to_datetime(df.crash_date.dt.strftime("%Y-%m-%d") + " " +
                                df.crash_time.astype(str), errors="coerce")
print("After join:", len(df))

After join: 74384


### 6. Check H2 fields: age, crash time, injury type
NYPD records sometimes use age 0 to mean "unknown," so we check how many pedestrians have age 0. If there are more than a handful, age 0 is treated as missing. Ages outside 0–110 are also set to missing. We then report the share of missing age and crash time and the distribution of injury types.

In [10]:
df["person_age"] = pd.to_numeric(df.person_age, errors="coerce")

n_zero = (df.person_age == 0).sum()
print("Pedestrians with age = 0:", n_zero)
if n_zero > 10:
    df.loc[df.person_age == 0, "person_age"] = np.nan
    print("-> age 0 treated as unknown (set to missing)")

df.loc[~df.person_age.between(0, 110), "person_age"] = np.nan

print("Missing age:       ", f"{df.person_age.isna().mean():.1%}")
print("Missing crash time:", f"{df.crash_dt.isna().mean():.1%}")
print("Injury types:\n", df.person_injury.value_counts())

Pedestrians with age = 0: 758
-> age 0 treated as unknown (set to missing)
Missing age:        2.4%
Missing crash time: 0.0%
Injury types:
 person_injury
Injured    73407
Killed       977
Name: count, dtype: int64


### 7. Missing coordinates
Records with missing coordinates or coordinates outside NYC cannot be linked to intersections, so they are dropped. We report the number and share dropped by year.

In [11]:
df["latitude"]  = pd.to_numeric(df.latitude, errors="coerce")
df["longitude"] = pd.to_numeric(df.longitude, errors="coerce")
valid = df.latitude.between(40.49, 40.92) & df.longitude.between(-74.26, -73.70)

missing_by_year = (~valid).groupby(df.crash_dt.dt.year).agg(["sum", "mean"])
missing_by_year.columns = ["n_missing", "share_missing"]
print(missing_by_year.round(3))
print(f"\nOverall: {(~valid).sum():,} of {len(df):,} ({(~valid).mean():.1%}) dropped for missing/invalid coordinates")

df = df[valid].copy()

          n_missing  share_missing
crash_dt                          
2016            999          0.118
2017            465          0.041
2018            391          0.035
2019            392          0.037
2020            302          0.044
2021            459          0.060
2022            704          0.077
2023            422          0.046

Overall: 4,134 of 74,384 (5.6%) dropped for missing/invalid coordinates


**Note on missing coordinates (limitation):** Missing locations are not evenly spread over time. In 2016, 11.8% of pedestrian injury records lacked valid coordinates, compared with 3.5–7.7% in 2017–2023. Injuries from 2016 are therefore more likely to be excluded from intersection linkage, which may undercount pre-installation injuries in the earliest year. Year fixed effects absorb citywide differences in this rate, but uneven missingness across intersections cannot be ruled out.

### 8. Create analysis flags
- **night** = 1 if the crash occurred between 6 pm and 6 am (fixed rule)
- **age_65plus** = 1 if the pedestrian was 65 or older (missing if age is unknown)
- **killed** = 1 if the pedestrian was killed
- **year** = crash year (used to build the intersection-year panel)

In [12]:
hour = df.crash_dt.dt.hour
df["night"]      = ((hour >= 18) | (hour < 6)).astype(int)
df["age_65plus"] = np.where(df.person_age.isna(), np.nan, (df.person_age >= 65).astype(int))
df["killed"]     = (df.person_injury == "Killed").astype(int)
df["year"]       = df.crash_dt.dt.year

print(df[["night", "age_65plus", "killed"]].mean().round(3))
print(df.year.value_counts().sort_index())

night         0.394
age_65plus    0.143
killed        0.013
dtype: float64
year
2016     7445
2017    10813
2018    10855
2019    10307
2020     6491
2021     7175
2022     8404
2023     8760
Name: count, dtype: int64


### 9. Save the clean pedestrian-injury file
The final file has one row per injured or killed pedestrian, with identifiers, timestamp, year, borough, coordinates, age, the 65+ indicator, injury type, the killed indicator, and the night indicator. It is shared with the team to link injuries to intersections.

In [13]:
cols = ["collision_id", "person_id", "crash_dt", "year", "borough", "latitude", "longitude",
        "person_age", "age_65plus", "person_injury", "killed", "night"]
out = df[cols]
out.to_csv("/content/clean_pedestrian_injuries.csv", index=False)
print("Saved:", out.shape)

Saved: (70250, 12)


In [14]:
from google.colab import files
files.download("/content/clean_pedestrian_injuries.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
print(f'''DATA COLLECTION & CLEANING SUMMARY (Person 1)
- Sources: NYC Open Data Motor Vehicle Collisions (Crashes and Person tables), {START} to {END}.
- Downloaded {len(crashes):,} crashes with at least one pedestrian injured or killed, and {len(persons):,} pedestrian person records.
- After the injury filter and de-duplication: {len(peds):,} pedestrians ({(peds.person_injury == "Injured").sum():,} injured, {(peds.person_injury == "Killed").sum():,} killed).
- {len(valid):,} pedestrian records joined to crashes on collision ID.
- Age = 0 found in {n_zero:,} records ({"treated as unknown" if n_zero > 10 else "kept"}); ages outside 0-110 set to missing. Missing age: {out.person_age.isna().mean():.1%}.
- {(~valid).sum():,} records ({(~valid).mean():.1%}) dropped for missing/invalid coordinates; share by year ranged from {missing_by_year.share_missing.min():.1%} to {missing_by_year.share_missing.max():.1%} (highest in {missing_by_year.share_missing.idxmax()}).
- Night flag (6 pm-6 am): {out.night.mean():.1%} of injuries. Age 65+: {out.age_65plus.mean():.1%} of injuries with known age. Killed: {out.killed.mean():.1%}.
- Final file clean_pedestrian_injuries.csv: {len(out):,} pedestrian injuries, {out.shape[1]} columns, {out.year.min()}-{out.year.max()}.''')

DATA COLLECTION & CLEANING SUMMARY (Person 1)
- Sources: NYC Open Data Motor Vehicle Collisions (Crashes and Person tables), 2016-04-01 to 2023-12-31.
- Downloaded 71,273 crashes with at least one pedestrian injured or killed, and 74,384 pedestrian person records.
- After the injury filter and de-duplication: 74,384 pedestrians (73,407 injured, 977 killed).
- 74,384 pedestrian records joined to crashes on collision ID.
- Age = 0 found in 758 records (treated as unknown); ages outside 0-110 set to missing. Missing age: 2.4%.
- 4,134 records (5.6%) dropped for missing/invalid coordinates; share by year ranged from 3.5% to 11.8% (highest in 2016).
- Night flag (6 pm-6 am): 39.4% of injuries. Age 65+: 14.3% of injuries with known age. Killed: 1.3%.
- Final file clean_pedestrian_injuries.csv: 70,250 pedestrian injuries, 12 columns, 2016-2023.


---
# PERSON 2 (Seerat Hameed): Treatment data and the intersection-year panel

**What this section does, in plain words.** Karuna's file is a list of injured pedestrians with a location and a date. To test whether APS make intersections safer, we need a table where **each row is one intersection in one year** with the number of injuries there that year, and whether that intersection had an APS that year. That table is called a **panel**. Building it takes six steps:

1. Download the APS, LPI and Barnes Dance location lists.
2. **Installs-per-year check:** count how many APS were installed each year, to decide between the main design and the fallback.
3. Check whether the LPI and Barnes Dance lists have install dates.
4. Count **mid-block** APS (ones not at a corner, like "150' West of ...") and leave them out.
5. Merge APS points less than 100 ft apart into one intersection (one intersection often has several APS buttons on different corners).
6. Attach every injury to the nearest APS intersection within 100 ft, then count injuries per intersection per year and save the panel.

**How to run:** run Karuna's cells above first (Runtime > Run all does everything in order). Each cell below prints a short check so you can see it worked.

In [15]:
# STEP 1: Download the three treatment datasets from NYC Open Data
!wget -q -O data/aps.csv    "https://data.cityofnewyork.us/resource/de3m-c5p4.csv?\$limit=200000"
!wget -q -O data/lpi.csv    "https://data.cityofnewyork.us/resource/xc4v-ntf4.csv?\$limit=200000"
!wget -q -O data/barnes.csv "https://data.cityofnewyork.us/resource/8kuj-2n3u.csv?\$limit=200000"
!wc -l data/aps.csv data/lpi.csv data/barnes.csv

   4405 data/aps.csv
   6811 data/lpi.csv
    738 data/barnes.csv
  11954 total


In [16]:
# Helper functions. Every NYC dataset names its columns differently, so these find
# the date column and the coordinates automatically instead of guessing names.
import re
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

STUDY_YEARS = list(range(2016, 2024))     # 2016-2023, same window as Karuna's file
RADIUS_FT   = 100                         # 100 ft, following Zadey, Morrison et al. (2025)

def to_feet(lon, lat):
    # Convert longitude/latitude to NY State Plane (EPSG:2263), which is measured in feet,
    # so that "100 ft" is an exact distance.
    lon, lat = np.asarray(lon, float), np.asarray(lat, float)
    try:
        from pyproj import Transformer
        t = Transformer.from_crs("EPSG:4326", "EPSG:2263", always_xy=True)
        x, y = t.transform(lon, lat)
        return np.column_stack([x, y])
    except ImportError:   # backup: flat-earth approximation, accurate to <1% at NYC scale
        ft_per_deg_lat = 364_000.0
        x = (lon + 74.0) * ft_per_deg_lat * np.cos(np.radians(40.7))
        y = (lat - 40.5) * ft_per_deg_lat
        return np.column_stack([x, y])

def find_date_col(d):
    # Pick the column that looks most like an install date.
    best, best_n = None, 0
    for c in d.columns:
        name = c.lower()
        if not any(k in name for k in ("date", "insta", "install", "year")):
            continue
        s = d[c]
        if s.notna().sum() == 0:
            continue
        if pd.api.types.is_numeric_dtype(s) and s.dropna().between(1950, 2035).all() and (s.dropna() % 1 == 0).all():
            parsed = pd.to_datetime(s.astype("Int64").astype(str), format="%Y", errors="coerce")
        else:
            parsed = pd.to_datetime(s.astype(str).str[:10], errors="coerce", format="mixed")
        ok = parsed.dt.year.between(1950, 2035).sum()
        if ok > best_n:
            best, best_n = c, ok
    return best

def parse_dates(d, col):
    s = d[col]
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_datetime(s.round().astype("Int64").astype(str), format="%Y", errors="coerce")
    return pd.to_datetime(s.astype(str).str[:10], errors="coerce", format="mixed")

def get_lonlat(d):
    # Return (lon, lat) arrays for each row, from whatever coordinate format the file uses.
    n = len(d)
    lon, lat = np.full(n, np.nan), np.full(n, np.nan)
    num = d.apply(pd.to_numeric, errors="coerce")
    # (a) separate longitude / latitude number columns
    lon_c = [c for c in num if num[c].notna().mean() > 0.5 and num[c].median() < -73.6 and num[c].median() > -74.3]
    lat_c = [c for c in num if num[c].notna().mean() > 0.5 and 40.4 < num[c].median() < 41.0]
    if lon_c and lat_c:
        lon, lat = np.array(num[lon_c[0]], dtype=float), np.array(num[lat_c[0]], dtype=float)
    # (b) a geometry column written like "POINT (-73.98 40.75)"
    for c in d.columns:
        if np.isnan(lon).mean() < 0.05:
            break
        s = d[c].astype(str)
        m = s.str.extract(r"POINT\s*\(\s*(-?\d+\.?\d*)\s+(-?\d+\.?\d*)", flags=re.I).astype(float)
        if m[0].notna().mean() > 0.5:
            fill = np.isnan(lon) & m[0].notna().to_numpy()
            lon[fill], lat[fill] = m[0].to_numpy()[fill], m[1].to_numpy()[fill]
            continue
        m = s.str.extract(r"\(\s*(40\.\d+)\s*,\s*(-7[34]\.\d+)\s*\)").astype(float)  # "(40.75, -73.98)"
        if m[0].notna().mean() > 0.5:
            fill = np.isnan(lon) & m[0].notna().to_numpy()
            lon[fill], lat[fill] = m[1].to_numpy()[fill], m[0].to_numpy()[fill]
    ok = (lon > -74.30) & (lon < -73.65) & (lat > 40.45) & (lat < 40.95)
    lon[~ok], lat[~ok] = np.nan, np.nan
    return lon, lat

def text_of(d):
    # All text columns glued together (street names, location descriptions).
    obj = [c for c in d.columns if not pd.api.types.is_numeric_dtype(d[c])]
    return d[obj].fillna("").astype(str).agg(" ".join, axis=1).str.lower()

aps_raw    = pd.read_csv("data/aps.csv", low_memory=False)
lpi_raw    = pd.read_csv("data/lpi.csv", low_memory=False)
barnes_raw = pd.read_csv("data/barnes.csv", low_memory=False)
for name, d in [("APS", aps_raw), ("LPI", lpi_raw), ("Barnes Dance", barnes_raw)]:
    print(f"{name}: {len(d):,} rows | columns: {list(d.columns)}")

APS: 4,404 rows | columns: ['the_geom', 'borocode', 'boroname', 'borocd', 'coundist', 'assemdist', 'stsendist', 'congdist', 'location', 'borough', 'date_insta', 'point_x', 'point_y', 'femafldz', 'femafldt', 'hrcevac', 'ntaname']
LPI: 6,810 rows | columns: ['the_geom', 'main_street', 'cross_stree', 'install_da', 'long', 'lat', 'x', 'y']
Barnes Dance: 737 rows | columns: ['the_geom', 'borocode', 'boroname', 'borocd', 'coundist', 'assemdist', 'stsendist', 'congdist', 'main_stree', 'cross_stre', 'barnes_dan', 'modified_b', 'mid_block_', 't_oneway_a', 't_oneway_1', 'installati', 'ntaname', 'femafldz', 'femafldt', 'hrcevac']


### STEP 2: Installs-per-year check

The main design needs enough intersections that got APS **during** the study (2018–2022, the *treated* group) and enough that got them **after** it (2024 or later, the *comparison* group). The cell counts both and prints the decision.

In [17]:
aps = aps_raw.copy()
aps_date_col = find_date_col(aps)
if aps_date_col is None:
    raise ValueError("No install-date column found in the APS file. Check the column list printed above.")
aps["install_date"] = parse_dates(aps, aps_date_col)
aps["install_year"] = aps.install_date.dt.year.astype("Int64")
aps["lon"], aps["lat"] = get_lonlat(aps)
print(f"APS install-date column used: '{aps_date_col}'")
print(f"APS rows missing an install date: {aps.install_year.isna().sum():,}")
print(f"APS rows missing coordinates:     {aps.lon.isna().sum():,}\n")

per_year = aps.install_year.value_counts().sort_index().astype(int)
print("APS installs per year (raw rows, before merging into intersections):")
print(per_year.to_string())

n_treat_raw = aps.install_year.between(2018, 2022).sum()
n_comp_raw  = (aps.install_year >= 2024).sum()
print(f"\nTreated-window rows (2018-2022): {n_treat_raw:,}")
print(f"Comparison rows (2024+):         {n_comp_raw:,}")

APS install-date column used: 'date_insta'
APS rows missing an install date: 13
APS rows missing coordinates:     0

APS installs per year (raw rows, before merging into intersections):
install_year
2010      3
2011      2
2012     27
2013     30
2014     23
2015     36
2016     75
2017     75
2018     89
2019    177
2020    203
2021    215
2022    519
2023    887
2024    866
2025    737
2026    427

Treated-window rows (2018-2022): 1,203
Comparison rows (2024+):         2,030


### STEP 3: Do the LPI and Barnes Dance lists have install dates?

If they do, we can mark the exact years each intersection had an LPI (`LPI_active`), which Wenjin can add to Model 1. If not, the LPI is just a yes/no flag, which the intersection fixed effects absorb anyway.

In [18]:
treat_info = {}
for name, d in [("LPI", lpi_raw), ("Barnes", barnes_raw)]:
    d = d.copy()
    col = find_date_col(d)
    d["lon"], d["lat"] = get_lonlat(d)
    if col is not None:
        d["year"] = parse_dates(d, col).dt.year
        share = d.year.notna().mean()
        yrs = f"years {int(d.year.min())}-{int(d.year.max())}" if d.year.notna().any() else "no readable years"
        print(f"{name}: install-date column = '{col}', {share:.0%} of rows have a date, {yrs}")
        has_dates = share > 0.5
    else:
        d["year"] = np.nan
        print(f"{name}: NO install-date column. Only a yes/no 'has {name}' flag will be used.")
        has_dates = False
    print(f"   rows with valid coordinates: {d.lon.notna().sum():,} of {len(d):,}")
    treat_info[name] = {"data": d[d.lon.notna()].reset_index(drop=True), "has_dates": has_dates}

LPI: install-date column = 'install_da', 100% of rows have a date, years 1976-2026
   rows with valid coordinates: 6,810 of 6,810
Barnes: install-date column = 'installati', 20% of rows have a date, years 2026-2026
   rows with valid coordinates: 737 of 737


### STEP 4: Count mid-block APS entries and leave them out

Some APS are not at a street corner (for example "Broadway between 72 St and 73 St" or "150' West of Main St"). These are **mid-block** crossings, not intersections, so they don't fit an intersection study. The cell counts them and, by default, removes them. If your group decides to keep them, change `EXCLUDE_MIDBLOCK` to `False`.

In [19]:
EXCLUDE_MIDBLOCK = True   # group decision: True = drop mid-block APS

midblock_pattern = (r"\bbetween\b|\bbtwn\b|\bb/w\b|mid-?block"
                    r"|\d+\s*(?:'|ft\b|feet\b)"                  # 150' / 150 ft / 150 feet
                    r"|\b(?:north|south|east|west)\s+of\b"         # West of
                    r"|\b[nsew]/o\b")                              # w/o, n/o
aps["midblock"] = text_of(aps_raw).str.contains(midblock_pattern, regex=True)
n_mid = int(aps.midblock.sum())
print(f"Mid-block APS entries: {n_mid:,} of {len(aps):,} ({n_mid/len(aps):.1%})")
print("Examples:")
print(text_of(aps_raw)[aps.midblock].head(5).str[:90].to_string())

n_start = len(aps)
aps = aps[aps.lon.notna() & aps.install_year.notna()]
aps["install_year"] = aps.install_year.astype(int)
if EXCLUDE_MIDBLOCK:
    aps = aps[~aps.midblock]
aps = aps.reset_index(drop=True)
print(f"\nAPS rows kept for analysis: {len(aps):,} of {n_start:,} "
      f"(dropped missing date/coordinates{' and mid-block' if EXCLUDE_MIDBLOCK else ''})")

Mid-block APS entries: 209 of 4,404 (4.7%)
Examples:
8      point (-74.10843852749848 40.569776775140255) ...
76     point (-74.169801506965 40.57576277620125) sta...
99     point (-73.78372518002162 40.6753548555387) qu...
104    point (-73.91067157404308 40.845493714546485) ...
109    point (-73.854670600339 40.70677273955001) que...

APS rows kept for analysis: 4,182 of 4,404 (dropped missing date/coordinates and mid-block)


### STEP 5: Merge APS points less than 100 ft apart into one intersection

One intersection usually has several APS buttons (one per corner or crosswalk), each listed as its own row. Points closer than 100 ft are joined into one intersection. The intersection's **install year is the first year any APS appeared there**, and its location is the average of its points.

In [20]:
xy = to_feet(aps.lon, aps.lat)
pairs = cKDTree(xy).query_pairs(r=RADIUS_FT, output_type="ndarray")
graph = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(len(aps), len(aps)))
n_int, labels = connected_components(graph, directed=False)
aps["intersection_id"] = labels

boro_col = next((c for c in aps_raw.columns if "boro" in c.lower()), None)
aps["aps_borough"] = (aps[boro_col].astype(str).str.strip().str.upper() if boro_col else np.nan)
boro_codes = {"1": "MANHATTAN", "2": "BRONX", "3": "BROOKLYN", "4": "QUEENS", "5": "STATEN ISLAND",
              "MN": "MANHATTAN", "BX": "BRONX", "BK": "BROOKLYN", "QN": "QUEENS", "SI": "STATEN ISLAND"}
aps["aps_borough"] = aps.aps_borough.replace(boro_codes)

inter = aps.groupby("intersection_id").agg(
    lon=("lon", "mean"), lat=("lat", "mean"),
    install_year=("install_year", "min"),
    n_aps_points=("lon", "size"),
    aps_borough=("aps_borough", lambda s: s.mode().iloc[0] if s.notna().any() else np.nan),
).reset_index()
inter["install_year"] = inter.install_year.astype(int)
inter[["x_ft", "y_ft"]] = to_feet(inter.lon, inter.lat)

print(f"{len(aps):,} APS points merged into {len(inter):,} intersections "
      f"({(inter.n_aps_points > 1).sum():,} had more than one point)")
print("\nAPS intersections by first install year:")
print(inter.install_year.value_counts().sort_index().to_string())

4,182 APS points merged into 4,177 intersections (5 had more than one point)

APS intersections by first install year:
install_year
2010      3
2011      1
2012     25
2013     24
2014     23
2015     36
2016     58
2017     59
2018     72
2019    150
2020    191
2021    210
2022    492
2023    850
2024    842
2025    715
2026    426


In [21]:
# DESIGN DECISION (main vs. fallback), now counted per intersection
inter["group"] = np.select(
    [inter.install_year.between(2018, 2022), inter.install_year >= 2024],
    ["treated", "comparison"], default="excluded")
n_treated = (inter.group == "treated").sum()
n_comp    = (inter.group == "comparison").sum()
MIN_PER_GROUP = 30
print(f"Treated intersections (APS installed 2018-2022): {n_treated:,}")
print(f"Comparison intersections (APS installed 2024+):  {n_comp:,}")
print(f"Excluded (installed before 2018, or in 2023):    {(inter.group == 'excluded').sum():,}")
DESIGN = "MAIN" if (n_treated >= MIN_PER_GROUP and n_comp >= MIN_PER_GROUP) else "FALLBACK"
print(f"\n>>> DECISION: {DESIGN} design "
      f"(rule: at least {MIN_PER_GROUP} intersections in each group). Share this with the group.")

Treated intersections (APS installed 2018-2022): 1,115
Comparison intersections (APS installed 2024+):  1,983
Excluded (installed before 2018, or in 2023):    1,079

>>> DECISION: MAIN design (rule: at least 30 intersections in each group). Share this with the group.


### STEP 6: Attach each injury to the nearest APS intersection within 100 ft

For every injured pedestrian in Karuna's file we find the closest APS intersection. If it is within 100 ft, the injury counts for that intersection; otherwise it is not near any APS intersection and is left out. We match against **all** APS intersections first, so an injury next to an excluded intersection isn't wrongly pushed to a farther one. The same 100 ft rule marks which intersections have an LPI or a Barnes Dance.

In [22]:
inj = pd.read_csv("/content/clean_pedestrian_injuries.csv", parse_dates=["crash_dt"])
inj["year"] = inj.crash_dt.dt.year
inj_xy = to_feet(inj.longitude, inj.latitude)

tree = cKDTree(inter[["x_ft", "y_ft"]].to_numpy())
dist, idx = tree.query(inj_xy, k=1, distance_upper_bound=RADIUS_FT)
matched = np.isfinite(dist)
inj["intersection_id"] = np.where(matched, inter.intersection_id.to_numpy()[np.minimum(idx, len(inter) - 1)], -1)
inj["dist_ft"] = np.where(matched, dist, np.nan)
inj_m = inj[matched].copy()

print(f"Injuries within {RADIUS_FT} ft of an APS intersection: {matched.sum():,} of {len(inj):,} ({matched.mean():.1%})")
print(f"Median distance to the intersection: {inj_m.dist_ft.median():.0f} ft")

# LPI and Barnes Dance: does each intersection have one within 100 ft? (and since when, if dated)
for name, col in [("LPI", "lpi"), ("Barnes", "barnes")]:
    t = treat_info[name]["data"]
    if len(t) == 0:
        inter[col], inter[f"{col}_year"] = 0, np.nan
        print(f"{name}: no usable coordinates, flag set to 0")
        continue
    t_xy = to_feet(t.lon, t.lat)
    d_t, i_t = cKDTree(t_xy).query(inter[["x_ft", "y_ft"]].to_numpy(), k=1, distance_upper_bound=RADIUS_FT)
    near = np.isfinite(d_t)
    inter[col] = near.astype(int)
    if treat_info[name]["has_dates"]:
        # earliest dated install among all points within 100 ft
        hits = cKDTree(t_xy).query_ball_point(inter[["x_ft", "y_ft"]].to_numpy(), r=RADIUS_FT)
        inter[f"{col}_year"] = [t.year.iloc[h].min() if len(h) else np.nan for h in hits]
    else:
        inter[f"{col}_year"] = np.nan
    print(f"Intersections with {name} within {RADIUS_FT} ft: {inter[col].sum():,} of {len(inter):,}")

Injuries within 100 ft of an APS intersection: 15,219 of 70,250 (21.7%)
Median distance to the intersection: 3 ft
Intersections with LPI within 100 ft: 2,411 of 4,177
Intersections with Barnes within 100 ft: 475 of 4,177


### STEP 7: Build the intersection × year panel

One row per study intersection per year (2016–2023), including years with **zero** injuries (those zeros matter: most intersection-years have no injuries). Rules from the project plan:

* Only **treated** (APS 2018–2022) and **comparison** (APS 2024+) intersections are kept.
* `APS_active` = 1 in years **after** the install year, 0 otherwise. Comparison intersections are always 0.
* The **install year itself is dropped** for treated intersections, since it is part before and part after.

In [23]:
study = inter[inter.group != "excluded"].copy()
grid = pd.MultiIndex.from_product([study.intersection_id, STUDY_YEARS],
                                  names=["intersection_id", "year"]).to_frame(index=False)

counts = (inj_m.groupby(["intersection_id", "year"])
          .agg(injuries=("killed", "size"),
               injuries_65plus=("age_65plus", "sum"),
               injuries_night=("night", "sum"),
               killed=("killed", "sum"))
          .reset_index())
panel = grid.merge(counts, on=["intersection_id", "year"], how="left")
count_cols = ["injuries", "injuries_65plus", "injuries_night", "killed"]
panel[count_cols] = panel[count_cols].fillna(0).astype(int)

# Borough: from the APS file if it has one, otherwise the most common borough of nearby crashes
crash_boro = (inj_m.dropna(subset=["borough"]).groupby("intersection_id").borough
              .agg(lambda s: s.mode().iloc[0]))
study["borough"] = study.aps_borough.where(study.aps_borough.isin(
    ["MANHATTAN", "BRONX", "BROOKLYN", "QUEENS", "STATEN ISLAND"]))
study["borough"] = study.borough.fillna(study.intersection_id.map(crash_boro)).fillna("UNKNOWN")

panel = panel.merge(study[["intersection_id", "group", "install_year", "borough", "lat", "lon",
                           "n_aps_points", "lpi", "lpi_year", "barnes", "barnes_year"]],
                    on="intersection_id", how="left")
panel["treated"]    = (panel.group == "treated").astype(int)
panel["APS_active"] = ((panel.treated == 1) & (panel.year > panel.install_year)).astype(int)
panel["any_injury"] = (panel.injuries > 0).astype(int)

# LPI / Barnes "active" in a year: since its install year if dated; if undated, the yes/no flag
for col in ["lpi", "barnes"]:
    dated = panel[f"{col}_year"].notna()
    panel[f"{col.upper()}_active"] = np.where(dated, (panel.year >= panel[f"{col}_year"]).astype(int), panel[col])
panel = panel.rename(columns={"lpi": "LPI", "barnes": "barnes_dance", "LPI_active": "LPI_active",
                              "BARNES_active": "barnes_active"})

# Drop the install year for treated intersections
n_before = len(panel)
panel = panel[~((panel.treated == 1) & (panel.year == panel.install_year))].reset_index(drop=True)
print(f"Dropped {n_before - len(panel):,} install-year rows for treated intersections")

panel = panel[["intersection_id", "year", "borough", "group", "treated", "install_year", "APS_active",
               "injuries", "injuries_65plus", "injuries_night", "killed", "any_injury",
               "LPI", "LPI_active", "barnes_dance", "barnes_active", "n_aps_points", "lat", "lon"]]

print(f"\nPanel: {len(panel):,} rows = {panel.intersection_id.nunique():,} intersections x up to {len(STUDY_YEARS)} years")
print(panel.groupby("group").agg(intersections=("intersection_id", "nunique"),
                                 rows=("year", "size"),
                                 mean_injuries=("injuries", "mean"),
                                 share_any_injury=("any_injury", "mean")).round(3))
print("\nRows with APS_active = 1:", int(panel.APS_active.sum()))
print("\nQuick checks (all should say True):")
print("  no duplicate intersection-years:", not panel.duplicated(["intersection_id", "year"]).any())
print("  comparison group never active:  ", panel.loc[panel.group == "comparison", "APS_active"].eq(0).all())
print("  no missing values in counts:    ", panel[count_cols].notna().all().all())
panel.head()

Dropped 1,115 install-year rows for treated intersections

Panel: 23,669 rows = 3,098 intersections x up to 8 years
            intersections   rows  mean_injuries  share_any_injury
group                                                            
comparison           1983  15864          0.454             0.295
treated              1115   7805          0.473             0.292

Rows with APS_active = 1: 2445

Quick checks (all should say True):
  no duplicate intersection-years: True
  comparison group never active:   True
  no missing values in counts:     True


,intersection_id,year,borough,group,treated,install_year,APS_active,injuries,injuries_65plus,injuries_night,killed,any_injury,LPI,LPI_active,barnes_dance,barnes_active,n_aps_points,lat,lon
0,1,2016,MANHATTAN,treated,1,2022,0,3,0,1,0,1,1,1,0,0,1,40.743783,-73.973511
1,1,2017,MANHATTAN,treated,1,2022,0,1,0,0,0,1,1,1,0,0,1,40.743783,-73.973511
2,1,2018,MANHATTAN,treated,1,2022,0,4,0,3,0,1,1,1,0,0,1,40.743783,-73.973511
3,1,2019,MANHATTAN,treated,1,2022,0,3,0,2,0,1,1,1,0,0,1,40.743783,-73.973511
4,1,2020,MANHATTAN,treated,1,2022,0,3,0,1,0,1,1,1,0,0,1,40.743783,-73.973511


### STEP 8: Save the panel and the data dictionary (the deliverables)

In [24]:
panel.to_csv("/content/aps_intersection_panel.csv", index=False)

dictionary = pd.DataFrame([
    ("intersection_id", "ID of the APS intersection (APS points within 100 ft merged into one)"),
    ("year", "Calendar year, 2016-2023 (2016 covers April-December only)"),
    ("borough", "Borough of the intersection"),
    ("group", "'treated' = first APS installed 2018-2022; 'comparison' = first APS installed 2024 or later"),
    ("treated", "1 if group is treated, else 0"),
    ("install_year", "Year the first APS was installed at the intersection"),
    ("APS_active", "1 in years after the APS install year (treated only), else 0. Install year itself is dropped"),
    ("injuries", "Pedestrians injured or killed within 100 ft of the intersection that year"),
    ("injuries_65plus", "Same, pedestrians aged 65+ (H2)"),
    ("injuries_night", "Same, crashes between 6 pm and 6 am (H2)"),
    ("killed", "Pedestrians killed within 100 ft that year"),
    ("any_injury", "1 if injuries > 0 (outcome for Model 2, logistic regression)"),
    ("LPI", "1 if a Leading Pedestrian Interval signal is within 100 ft"),
    ("LPI_active", "1 if the LPI was installed by that year (equals LPI if the LPI file has no dates)"),
    ("barnes_dance", "1 if an Exclusive Pedestrian Signal (Barnes Dance) is within 100 ft"),
    ("barnes_active", "1 if the Barnes Dance was installed by that year (equals barnes_dance if undated)"),
    ("n_aps_points", "Number of APS records merged into this intersection"),
    ("lat", "Latitude of the intersection (average of its APS points), for maps"),
    ("lon", "Longitude of the intersection, for maps"),
], columns=["column", "meaning"])
dictionary.to_csv("/content/panel_data_dictionary.csv", index=False)
print(dictionary.to_string(index=False))

from google.colab import files
files.download("/content/aps_intersection_panel.csv")
files.download("/content/panel_data_dictionary.csv")

         column                                                                                      meaning
intersection_id                        ID of the APS intersection (APS points within 100 ft merged into one)
           year                                   Calendar year, 2016-2023 (2016 covers April-December only)
        borough                                                                  Borough of the intersection
          group  'treated' = first APS installed 2018-2022; 'comparison' = first APS installed 2024 or later
        treated                                                                1 if group is treated, else 0
   install_year                                         Year the first APS was installed at the intersection
     APS_active 1 in years after the APS install year (treated only), else 0. Install year itself is dropped
       injuries                    Pedestrians injured or killed within 100 ft of the intersection that year
injuries_65plus    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Summary numbers for Slide 3 and the report (auto-filled)

In [25]:
print(f'''DATA PREPARATION SUMMARY (Person 2)
- APS dataset: {len(aps_raw):,} records. {n_mid:,} were mid-block and {"excluded" if EXCLUDE_MIDBLOCK else "kept"}; records missing an install date or coordinates were dropped.
- The remaining {len(aps):,} APS points were merged into {len(inter):,} intersections (points less than {RADIUS_FT} ft apart = one intersection).
- Treated intersections (first APS 2018-2022): {n_treated:,}. Comparison intersections (first APS 2024+): {n_comp:,}. Design: {DESIGN}.
- LPI dates available: {treat_info["LPI"]["has_dates"]}. Barnes Dance dates available: {treat_info["Barnes"]["has_dates"]}.
- {matched.sum():,} of {len(inj):,} pedestrian injuries ({matched.mean():.1%}) fell within {RADIUS_FT} ft of an APS intersection (nearest-intersection rule).
- Final panel: {len(panel):,} intersection-years, {panel.intersection_id.nunique():,} intersections, {panel.year.min()}-{panel.year.max()}; install year dropped for treated intersections.
- {1 - panel.any_injury.mean():.1%} of intersection-years have zero injuries (why accuracy is misleading for Model 2).''')

DATA PREPARATION SUMMARY (Person 2)
- APS dataset: 4,404 records. 209 were mid-block and excluded; records missing an install date or coordinates were dropped.
- The remaining 4,182 APS points were merged into 4,177 intersections (points less than 100 ft apart = one intersection).
- Treated intersections (first APS 2018-2022): 1,115. Comparison intersections (first APS 2024+): 1,983. Design: MAIN.
- LPI dates available: True. Barnes Dance dates available: False.
- 15,219 of 70,250 pedestrian injuries (21.7%) fell within 100 ft of an APS intersection (nearest-intersection rule).
- Final panel: 23,669 intersection-years, 3,098 intersections, 2016-2023; install year dropped for treated intersections.
- 70.6% of intersection-years have zero injuries (why accuracy is misleading for Model 2).
